In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_parquet('../data/processed/bank_marketing_primary.parquet')
print(f'Customers: {len(df):,}, Conversion: {df["y"].mean():.2%}')

In [ ]:
# Arm = canal de contato real (0 = cellular, 1 = telephone). Sem canais sinteticos aqui:
# a comparacao da Etapa 3 usa so o dado real observado.
df['arm'] = df['contact'].map({'cellular': 0, 'telephone': 1})
print('Arms assigned (0=cellular, 1=telephone):')
print(df['arm'].value_counts().sort_index())

In [ ]:
baseline_rate = df['y'].mean()
baseline_conversions = (df['y'] == 1).sum()
print(f'Baseline Conversion Rate: {baseline_rate:.4f} ({baseline_rate:.2%})')
print(f'Baseline Conversions: {int(baseline_conversions)} / {len(df)}')

In [ ]:
class BetaBernoulliBandit:
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)
        self.beta = np.ones(n_arms)
        self.trials = np.zeros(n_arms)
        self.successes = np.zeros(n_arms)
    def select_arm(self):
        thetas = [np.random.beta(self.alpha[i], self.beta[i]) for i in range(len(self.alpha))]
        return np.argmax(thetas)
    def update(self, arm, reward):
        if reward == 1:
            self.alpha[arm] += 1
        else:
            self.beta[arm] += 1
        self.trials[arm] += 1
        self.successes[arm] += reward

print('Thompson Sampling Bandit defined')

In [ ]:
# Taxa de conversao REAL medida por braco (nao inventada) - usada como a
# probabilidade verdadeira de cada braco na simulacao do bandit.
arm_true_rates = df.groupby('arm')['y'].mean().to_dict()
print('Taxa real por braco:', {k: f'{v:.2%}' for k, v in arm_true_rates.items()})

np.random.seed(42)
bandit = BetaBernoulliBandit(len(arm_true_rates))
print('Running Thompson Sampling simulation...')
for idx in range(len(df)):
    arm = bandit.select_arm()
    reward = 1 if np.random.random() < arm_true_rates[arm] else 0
    bandit.update(arm, reward)
    if (idx + 1) % 10000 == 0:
        print(f'  Processed {idx + 1:,} customers')

thompson_rate = bandit.successes.sum() / len(df)
thompson_conversions = int(bandit.successes.sum())
print(f'\nThompson Conversion Rate: {thompson_rate:.4f} ({thompson_rate:.2%})')
print(f'Thompson Conversions: {thompson_conversions} / {len(df)}')
print(f'Trials per arm: {dict(enumerate(bandit.trials.astype(int)))}')

In [ ]:
print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)
print(f'Baseline:  {baseline_rate:.4f} ({baseline_rate:.2%})')
print(f'Thompson:  {thompson_rate:.4f} ({thompson_rate:.2%})')
print(f'Improvement: {thompson_rate - baseline_rate:+.4f} ({(thompson_rate - baseline_rate)*100:+.2f}%)')
print('='*60)
if thompson_rate > baseline_rate:
    print('SUCCESS: Thompson Sampling OUTPERFORMED Baseline!')
else:
    print('Baseline performed better or equal')